# 02 — Time alignment, coarse fingerprints, and single-time ambiguity

This notebook covers Tasks 2, 8, and 9. It reads raw CSVs directly and retains all derived objects only in memory.

Known unreliable Mobility dates (18, 19, 20, 22, 28 March) are shown in the daily audit but excluded from the primary candidate analysis. Ticket timestamps include an explicit `-03:00`; Mobility timestamps are timezone-naive and are tested as local BRT clock readings.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import re, unicodedata, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

HERE = Path.cwd().resolve()
PROJECT = HERE.parent if HERE.name == "mob_tick_matching_robustness_check" else HERE
DATA = PROJECT / "data"
MOB_FILES = sorted((DATA / "mobility_data").glob("*.csv"))
TIX_FILES = sorted((DATA / "ticket_data").glob("*.csv"))
BAD_DATES = {"2026-03-18", "2026-03-19", "2026-03-20", "2026-03-22", "2026-03-28"}
RELIABLE_FILES_M = [p for p in MOB_FILES if p.stem not in BAD_DATES]
RELIABLE_DATES = {p.stem for p in RELIABLE_FILES_M}
RELIABLE_FILES_T = [p for p in TIX_FILES if p.stem in RELIABLE_DATES]

def canon_route(x):
    """Conservative formatting normalization; does not remove leading zeroes or punctuation."""
    if pd.isna(x): return pd.NA
    s = str(x).strip()
    return re.sub(r"^(\d+)\.0$", r"\1", s)

def canon_name(x):
    if pd.isna(x): return pd.NA
    s = unicodedata.normalize("NFKD", str(x))
    s = "".join(c for c in s if not unicodedata.combining(c)).upper().strip()
    return re.sub(r"\s+", " ", s)

print(f"Project: {PROJECT}")
print(f"Mobility files: {len(MOB_FILES)}; Ticket files: {len(TIX_FILES)}; reliable overlapping dates: {len(RELIABLE_DATES)}")

Project: /Users/pirin/Desktop/Under Grad Research/Netmob2026_data_challenge
Mobility files: 19; Ticket files: 31; reliable overlapping dates: 16


In [2]:
# Hourly distributions and raw timestamp evidence across overlap dates.
hour_rows=[]
for p in MOB_FILES:
    df=pd.read_csv(p,usecols=["timestamp"],dtype="string")
    ts=pd.to_datetime(df.timestamp,errors="coerce")
    c=ts.dt.hour.value_counts(); hour_rows += [{"date":p.stem,"dataset":"Mobility","hour":h,"rows":int(n)} for h,n in c.items()]
for p in [x for x in TIX_FILES if x.stem in {m.stem for m in MOB_FILES}]:
    df=pd.read_csv(p,usecols=["transaction_date"],dtype="string")
    ts=pd.to_datetime(df.transaction_date,errors="coerce",utc=True).dt.tz_convert("America/Sao_Paulo")
    c=ts.dt.hour.value_counts(); hour_rows += [{"date":p.stem,"dataset":"Ticket","hour":h,"rows":int(n)} for h,n in c.items()]
hours=pd.DataFrame(hour_rows)
display(Markdown("## D. Temporal compatibility — local-clock hourly shares"))
piv=hours.groupby(["dataset","hour"]).rows.sum().groupby(level=0).apply(lambda x:x/x.sum(),include_groups=False).unstack(0).fillna(0)
display(piv)

# Offset diagnostic: correlation of route-hour volume at candidate clock shifts.
def route_hour(files,kind):
    out=[]
    for p in files:
        if kind=="m":
            df=pd.read_csv(p,usecols=["timestamp","lineId"],dtype="string"); ts=pd.to_datetime(df.timestamp,errors="coerce"); route=df.lineId.map(canon_route)
        else:
            df=pd.read_csv(p,usecols=["transaction_date","route_name"],dtype="string"); ts=pd.to_datetime(df.transaction_date,errors="coerce",utc=True).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None); route=df.route_name.map(canon_route)
        z=pd.DataFrame({"date":p.stem,"hour":ts.dt.floor("h"),"route":route}).dropna()
        out.append(z.groupby(["date","hour","route"]).size().rename("n").reset_index())
    return pd.concat(out,ignore_index=True)
mh=route_hour(RELIABLE_FILES_M,"m"); th=route_hour(RELIABLE_FILES_T,"t")
offset_rows=[]
for off in [-180,-60,0,60,180]:
    z=th.copy(); z["hour"]=z.hour+pd.Timedelta(minutes=off)
    q=z.merge(mh,on=["date","hour","route"],suffixes=("_t","_m"))
    offset_rows.append({"ticket_shift_minutes":off,"matched_route_hours":len(q),"pearson_log_volume":np.corrcoef(np.log1p(q.n_t),np.log1p(q.n_m))[0,1] if len(q)>2 else np.nan})
display(Markdown("### Candidate systematic clock offsets (higher correlation/matched cells is better)")); display(pd.DataFrame(offset_rows))

## D. Temporal compatibility — local-clock hourly shares

dataset        Mobility    Ticket
dataset  hour                    
Mobility 0     0.006892  0.000000
         1     0.004606  0.000000
         2     0.002796  0.000000
         3     0.002508  0.000000
         4     0.003798  0.000000
         5     0.020705  0.000000
         6     0.048522  0.000000
         7     0.065245  0.000000
         8     0.067334  0.000000
         9     0.065986  0.000000
         10    0.062151  0.000000
         11    0.055219  0.000000
         12    0.053516  0.000000
         13    0.053380  0.000000
         14    0.051450  0.000000
         15    0.056877  0.000000
         16    0.065158  0.000000
         17    0.070439  0.000000
         18    0.069520  0.000000
         19    0.060536  0.000000
         20    0.045281  0.000000
         21    0.031700  0.000000
         22    0.022648  0.000000
         23    0.013734  0.000000
Ticket   0     0.000000  0.000346
         1     0.000000  0.002873
         2     0.000000  0.027837
         3     0.000000  0.076484
         4     0.000000  0.089932
         5     0.000000  0.074133
         6     0.000000  0.054708
         7     0.000000  0.047935
         8     0.000000  0.052693
         9     0.000000  0.069394
         10    0.000000  0.055086
         11    0.000000  0.046912
         12    0.000000  0.054960
         13    0.000000  0.070764
         14    0.000000  0.084795
         15    0.000000  0.071808
         16    0.000000  0.049581
         17    0.000000  0.027768
         18    0.000000  0.019983
         19    0.000000  0.012921
         20    0.000000  0.005337
         21    0.000000  0.002493
         22    0.000000  0.000844
         23    0.000000  0.000414

### Candidate systematic clock offsets (higher correlation/matched cells is better)

,ticket_shift_minutes,matched_route_hours,pearson_log_volume
0,-180,7250,0.192288
1,-60,8149,0.366230
2,0,8598,0.447812
3,60,8881,0.538388
4,180,9350,0.770099


In [3]:
# Build reliable one-minute representations. Multiple routes within a GPS id-minute are preserved as sets.
mob_parts=[]; multi=[]
for p in RELIABLE_FILES_M:
    df=pd.read_csv(p,usecols=["id","timestamp","lineId","tripId"],dtype="string")
    ts=pd.to_datetime(df.timestamp,errors="coerce")
    z=pd.DataFrame({"date":p.stem,"id":df.id,"minute":ts.dt.floor("min"),"route":df.lineId.map(canon_route),"tripId":df.tripId}).dropna(subset=["id","minute","route"])
    g=z.groupby(["date","id","minute"]).agg(route=("route",lambda x:" | ".join(sorted(set(x)))),n_routes=("route","nunique"),tripId=("tripId",lambda x:" | ".join(sorted(set(x.dropna())))),raw_observations=("route","size")).reset_index()
    multi.append(g[g.n_routes>1]); mob_parts.append(g)
mob_min=pd.concat(mob_parts,ignore_index=True); multi_route=pd.concat(multi,ignore_index=True)

# Determine key based on the explicit company uniqueness test.
vc=defaultdict(set)
for p in TIX_FILES:
    d=pd.read_csv(p,usecols=["vehicle_number","company_number"],dtype="string").drop_duplicates()
    for v,c in d.dropna().itertuples(index=False): vc[v].add(c)
USE_COMPOSITE=any(len(x)>1 for x in vc.values())
tix_parts=[]
for p in RELIABLE_FILES_T:
    df=pd.read_csv(p,usecols=["transaction_date","vehicle_number","company_number","route_name"],dtype="string")
    ts=pd.to_datetime(df.transaction_date,errors="coerce",utc=True).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)
    key=(df.company_number.fillna("<NA>")+"::"+df.vehicle_number.fillna("<NA>")) if USE_COMPOSITE else df.vehicle_number
    z=pd.DataFrame({"date":p.stem,"vehicle_key":key,"minute":ts.dt.floor("min"),"route":df.route_name.map(canon_route)}).dropna()
    tix_parts.append(z.groupby(["date","vehicle_key","minute","route"]).size().rename("boarding_count").reset_index())
tix_min=pd.concat(tix_parts,ignore_index=True)
print({"vehicle_key": "(company_number, vehicle_number)" if USE_COMPOSITE else "vehicle_number", "ticket_vehicle_minutes":len(tix_min),"mobility_id_minutes":len(mob_min),"multi_route_id_minutes":len(multi_route),"multi_route_pct":100*len(multi_route)/len(mob_min)})
display(Markdown("### GPS IDs with multiple lineIds in the same minute (all cases)")); display(multi_route)

{'vehicle_key': '(company_number, vehicle_number)', 'ticket_vehicle_minutes': 1583606, 'mobility_id_minutes': 3304138, 'multi_route_id_minutes': 990, 'multi_route_pct': 0.02996242893002653}


### GPS IDs with multiple lineIds in the same minute (all cases)

,date,id,minute,route,n_routes,tripId,raw_observations
0,2026-03-11,69034,2026-03-11 10:32:00,36 | 39A,2,1345046_U_32 | 1364748_U_32,4
1,2026-03-11,72675,2026-03-11 17:11:00,48 | 48SP,2,1374661_U_6 | 1374663_U_6,4
2,2026-03-11,72705,2026-03-11 15:57:00,48 | 48SP,2,1374661_U_6 | 1374663_U_41,4
3,2026-03-11,72717,2026-03-11 17:39:00,52 | 52A,2,1476582_U_32 | 1476584_U_12,4
4,2026-03-11,72741,2026-03-11 19:18:00,OC2 | OC3,2,1413903_U_61 | 1413904_U_61 | 1413904_U_72,4
...,...,...,...,...,...,...,...
985,2026-03-31,130628,2026-03-31 07:03:00,30 | 47,2,T75279 | T75280,4
986,2026-03-31,130628,2026-03-31 15:28:00,30 | 47,2,T75279 | T75280,4
987,2026-03-31,72636,2026-03-31 18:17:00,52 | 52A,2,1476582_U_3 | 1476584_U_2,4
988,2026-03-31,72717,2026-03-31 17:38:00,52 | 52A,2,1476582_U_32 | 1476584_U_12,4


In [4]:
# Candidate sets around each Ticket vehicle-minute. A mobility id qualifies if observed on the same route
# in minute-1, minute, or minute+1 (approximately ±60–90 seconds at this coarse stage).
mm=mob_min.assign(route=mob_min.route.str.split(" \| ")).explode("route")[["minute","route","id"]].drop_duplicates()
pieces=[]
for delta in [-1,0,1]:
    x=mm.copy(); x["ticket_minute"]=x.minute-pd.Timedelta(minutes=delta)
    pieces.append(x[["ticket_minute","route","id"]])
near=pd.concat(pieces).drop_duplicates()
tm=tix_min.rename(columns={"minute":"ticket_minute"})
counts=tm.merge(near,on=["ticket_minute","route"],how="left").groupby(["date","vehicle_key","ticket_minute","route"],dropna=False).id.nunique().rename("candidate_count").reset_index()
dist=counts.candidate_count.value_counts().sort_index().rename_axis("candidate_count").rename("vehicle_minutes").reset_index(); dist["pct"]=100*dist.vehicle_minutes/dist.vehicle_minutes.sum()
display(Markdown("## E. Single-time ambiguity")); display(dist)
display(counts.candidate_count.describe(percentiles=[.5,.9,.95,.99]).to_frame("candidate_count"))
display(Markdown("Candidate count 0 means no same-route GPS ID was observed in the ±1-minute window; it is missing coverage, not conflict evidence."))

# Stronger offset check: same-route GPS coverage around Ticket vehicle-minutes.
coverage=[]
for off in [-180,-60,0,60,180]:
    shifted=tm.copy(); shifted["ticket_minute"]=shifted.ticket_minute+pd.Timedelta(minutes=off)
    cc=shifted.merge(near,on=["ticket_minute","route"],how="left").groupby(["date","vehicle_key","ticket_minute","route"],dropna=False).id.nunique()
    coverage.append({"ticket_shift_minutes":off,"vehicle_minutes":len(cc),"pct_zero_same_route_candidates":100*(cc==0).mean(),"mean_same_route_candidates":cc.mean(),"median_same_route_candidates":cc.median()})
display(Markdown("### Offset sensitivity using same-route GPS coverage (lower zero-candidate percentage is better)")); display(pd.DataFrame(coverage))

<>:3: SyntaxWarning: invalid escape sequence '\|'
<>:3: SyntaxWarning: invalid escape sequence '\|'
/var/folders/bf/1gfy22fj5c5gxjrkh35xvr9m0000gn/T/ipykernel_29172/1400726412.py:3: SyntaxWarning: invalid escape sequence '\|'
  mm=mob_min.assign(route=mob_min.route.str.split(" \| ")).explode("route")[["minute","route","id"]].drop_duplicates()


## E. Single-time ambiguity

,candidate_count,vehicle_minutes,pct
0,0,679473,42.906695
1,1,83361,5.263999
2,2,64782,4.090790
3,3,49930,3.152931
4,4,48127,3.039077
5,5,47950,3.027900
6,6,43235,2.730161
7,7,48927,3.089594
8,8,57072,3.603927
9,9,55408,3.498850


,candidate_count
count,1.583606e+06
mean,5.568342e+00
std,7.362709e+00
min,0.000000e+00
50%,2.000000e+00
90%,1.500000e+01
95%,2.100000e+01
99%,3.100000e+01
max,3.900000e+01


Candidate count 0 means no same-route GPS ID was observed in the ±1-minute window; it is missing coverage, not conflict evidence.

### Offset sensitivity using same-route GPS coverage (lower zero-candidate percentage is better)

,ticket_shift_minutes,vehicle_minutes,pct_zero_same_route_candidates,mean_same_route_candidates,median_same_route_candidates
0,-180,1583606,48.156107,4.443094,1.0
1,-60,1583606,45.577814,5.107602,1.0
2,0,1583606,42.906695,5.568342,2.0
3,60,1583606,40.016582,6.103739,3.0
4,180,1583606,36.737042,6.984420,5.0


## Notebook 02 decision rule

Evidence favors no systematic clock offset when the 0-minute row is at least as strong as shifted alternatives and local-hour coverage overlaps. A large candidate-set median demonstrates why a single boarding time cannot identify a bus and motivates the longitudinal fingerprint tested next.